# 01 — Data Preparation & Validation

Heavy work is done by **`rebuild_paper_data.py`** (run from this directory).
This notebook validates the outputs and shows coverage summaries.

```bash
py rebuild_paper_data.py
```

## Finetuning modes in analysis
| Mode | Description |
|------|-------------|
| `no_finetune` | Global model on all 750 series (baseline) |
| `finetune_bucket_low` | Global model trained only on Low-bucket series (~250) |
| `finetune_bucket_medium` | Global model trained only on Medium-bucket series (~250) |
| `finetune_bucket_high` | Global model trained only on High-bucket series (~250) |



In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("data")
files = ["master_metrics.parquet","stl_delta.parquet","bucket_ft_delta.parquet",
         "features.parquet","bucket_manifest.parquet"]
print("=== Data files ===")
for f in files:
    p = DATA / f
    status = f"{p.stat().st_size/1e6:.1f} MB  [OK]" if p.exists() else "MISSING — run rebuild_paper_data.py"
    print(f"  {f:<40} {status}")

stale = DATA / "finetune_delta.parquet"
if stale.exists():
    print(f"  finetune_delta.parquet  [stale — delete and re-run rebuild]")


=== Data files ===
  master_metrics.parquet                   232.4 MB  [OK]
  stl_delta.parquet                        194.7 MB  [OK]
  bucket_ft_delta.parquet                  124.0 MB  [OK]
  features.parquet                         0.2 MB  [OK]
  bucket_manifest.parquet                  0.2 MB  [OK]


In [2]:
master = pd.read_parquet(DATA / "master_metrics.parquet")
print("=== Master metrics ===")
print(f"Total rows: {len(master):,}")
print("\nBy finetuning_mode:")
print(master["finetuning_mode"].value_counts().to_string())
print("\nBy model_family:")
print(master["model_family"].value_counts().to_string())
print("\nBy frequency:")
print(master["frequency"].value_counts().to_string())


=== Master metrics ===
Total rows: 3,382,722

By finetuning_mode:
finetuning_mode
no_finetune               1932984
finetune_bucket_low        610119
finetune_bucket_high       434889
finetune_bucket_medium     404730

By model_family:
model_family
mlforecast        1288656
neuralforecast     966492
transformers       644328
statistical        483246

By frequency:
frequency
monthly      1701000
quarterly    1681722


In [3]:
print("=== POCID coverage ===")
print(f"  not-null: {master['pocid'].notna().mean()*100:.1f}%  ({master['pocid'].notna().sum():,} rows)")
print()
print("=== Baseline stats (no_finetune) ===")
base = master[master["finetuning_mode"]=="no_finetune"]
print(base[["rel_naive_clipped","rel_naive_unclipped","mae","smape"]].describe().round(4).to_string())


=== POCID coverage ===
  not-null: 100.0%  (3,382,722 rows)

=== Baseline stats (no_finetune) ===
       rel_naive_clipped  rel_naive_unclipped           mae         smape
count       1.932984e+06         1.932984e+06  1.932984e+06  1.932984e+06
mean        1.040700e+00         1.044800e+00  6.778487e+02  1.441000e-01
std         6.105000e-01         7.297000e-01  9.101534e+02  1.569000e-01
min         1.870000e-02         1.870000e-02  9.587000e-01  8.000000e-04
25%         7.976000e-01         7.976000e-01  1.798585e+02  4.140000e-02
50%         9.875000e-01         9.875000e-01  4.062010e+02  9.030000e-02
75%         1.065900e+00         1.065900e+00  8.349730e+02  1.884000e-01
max         1.000000e+01         1.110049e+02  3.837143e+04  1.999200e+00


In [4]:
stl = pd.read_parquet(DATA / "stl_delta.parquet")
print("=== STL delta ===")
print(f"Rows: {len(stl):,}")
noft = stl[stl["finetuning_mode"]=="no_finetune"]
print("\nBy decomposition (no_finetune only):")
print(noft.groupby("decomposition_method")["rn_delta"]
      .agg(mean="mean",median="median",win_rate=lambda x:(x>0).mean()).round(4).to_string())


=== STL delta ===
Rows: 2,255,148

By decomposition (no_finetune only):
                            mean  median  win_rate
decomposition_method                              
stl_model_all_components -0.0939  0.0288    0.5324
stl_seasonal_naive       -0.0837  0.0051    0.5095


In [5]:
bkt = pd.read_parquet(DATA / "bucket_ft_delta.parquet")
print("=== Bucket finetuning delta ===")
print(f"Rows: {len(bkt):,}")
print()
print("By bucket × family (median rn_delta_ft):")
print(bkt.groupby(["trained_on_bucket","model_family"])["rn_delta_ft"]
      .agg(median="median",win_rate=lambda x:(x>0).mean()).round(4).to_string())
print()
print(f"Skipped tasks (no series in bucket): {bkt['rn_delta_ft'].isna().sum()}")


=== Bucket finetuning delta ===
Rows: 1,449,738

By bucket × family (median rn_delta_ft):
                                  median  win_rate
trained_on_bucket model_family                    
High              mlforecast      0.0540    0.5560
                  neuralforecast  0.0461    0.5623
                  transformers    0.0349    0.5560
Low               mlforecast      0.0627    0.5640
                  neuralforecast  0.0466    0.5560
                  transformers    0.0294    0.5467
Medium            mlforecast      0.0783    0.5764
                  neuralforecast  0.0581    0.5695
                  transformers    0.0337    0.5504

Skipped tasks (no series in bucket): 0


In [6]:
bkt_man = pd.read_parquet(DATA / "bucket_manifest.parquet")
print("=== Bucket manifest ===")
print(f"Shape: {bkt_man.shape}")
print()
print("Bucket sizes per feature × frequency:")
pivot = bkt_man.groupby(["feature_name","frequency","feature_tercile_bucket"]).size().unstack(fill_value=0)
print(pivot.to_string())
print()
print("NOTE: feature_nonlinearity has no Medium bucket (81-93% exact zeros → binary distribution).")


=== Bucket manifest ===
Shape: (18000, 8)

Bucket sizes per feature × frequency:
feature_tercile_bucket                       High   Low  Medium
feature_name                      frequency                    
feature_arch_stat                 monthly     500   500     500
                                  quarterly   500   500     500
feature_evolving_seasonality      monthly     500   500     500
                                  quarterly   500   500     500
feature_non_normality             monthly     500   500     500
                                  quarterly   500   500     500
feature_nonlinearity              monthly     273  1227       0
                                  quarterly   108  1392       0
feature_spectral_entropy          monthly     500   500     500
                                  quarterly   500   500     500
feature_structural_break_strength monthly     500   500     500
                                  quarterly   500   500     500

NOTE: feature_nonlinea

In [7]:
print("=== Task coverage ===")
n_base_configs = 6 * 2 * 3  # features × freqs × decomps
print(f"Base configs: {n_base_configs} (6 features × 2 freqs × 3 decomps)")
noft_tasks = master[master["finetuning_mode"]=="no_finetune"].groupby(
    ["feature_name","frequency","decomposition_method","model_family","model_name"]).ngroups
print(f"no_finetune tasks: {noft_tasks}  (expected {n_base_configs*12})")
bkt_tasks = bkt.groupby(
    ["feature_name","frequency","decomposition_method","model_family","model_name","trained_on_bucket"]).ngroups
expected_bkt = n_base_configs * 9 * 3  # 9 non-stat models × 3 buckets
print(f"bucket tasks: {bkt_tasks}  (expected ~{expected_bkt} minus empty nonlinearity-medium)")


=== Task coverage ===
Base configs: 36 (6 features × 2 freqs × 3 decomps)
no_finetune tasks: 432  (expected 432)
bucket tasks: 918  (expected ~972 minus empty nonlinearity-medium)
